# Multi-dataset PFN runs from data/ folder (bootstrap import)

This variant bootstraps `sys.path` to import the local `med3pipe` package without installing.

In [ ]:
import sys
from pathlib import Path

def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            print('Added repo root to sys.path:', base)
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

_add_repo_root_to_sys_path()


Now import and run the multi-dataset folder-driven pipeline.

In [ ]:
from pathlib import Path
from med3pipe.pipelines import run_multi_from_folder
from med3pipe.tabular.localpfn import LocalPFNConfig

# Configure which methods to run per dataset
methods = ("tabpfn", "localpfn")  # choose one or both

# Where to discover datasets and where to store outputs
datasets_dir = Path("data")
outputs_base_dir = Path("notebooks")

# Optionally restrict to a subset, e.g., ('gist', 'lipo')
dataset_names = None

# Optional: LoCalPFN overrides
local_cfg = LocalPFNConfig(
    k=8,                 # auto based on train size if None
    metric="euclidean",
    fit_adapter=True,      # set True to enable lightweight adapter fine-tuning
)

res = run_multi_from_folder(
    datasets_dir=datasets_dir,
    methods=methods,
    dataset_names=dataset_names,
    outputs_base_dir=outputs_base_dir,
    # Shared overrides (optional)
    sam3d_root=None,        # auto-detect if None
    model_type="vit_b_ori",
    checkpoint=None,
    device=None,            # "cuda" or "cpu"; auto if None
    n_components_max=500,
    random_state=42,
    # LoCalPFN
    local_cfg=local_cfg,
    # Discovery behavior
    require_sheet_csv=True,
    default_case_suffix="_CT",
    # Summary output
    save_summary=True,
)
res['summary_df']


In [ ]:
# Print where the summary CSV is written and preview a few rows
summary_path = res.get('summary_path')
print('Summary CSV:', summary_path)
try:
    import pandas as pd
    df = res['summary_df']
    display(df.head(20))
except Exception as e:
    print('Could not display summary:', e)
